# Notebook 04 — SHAP Value Computation

**Project:** Calibrated and Stability-Aware Explainable Intrusion Detection
**Author:** Md Anas Biswas, University of Portsmouth
**Stage:** 4 of 10

## What this notebook does

Computes SHAP feature attributions for all 6 canonical models, on the **eval-half** samples from Notebook 03 (so SHAP values align with calibrated probabilities).

## Method per model

| Model type | SHAP method | Why |
|---|---|---|
| Random Forest | `shap.TreeExplainer` | Fast, exact, polynomial time |
| XGBoost | `shap.TreeExplainer` | Same — built into XGBoost |
| DNN (MLP) | `shap.DeepExplainer` | Approximates Shapley values for deep nets via DeepLIFT |

## What gets saved

```
shap_values/nsl_kdd/
├── rf_binary_cw_shap.npy           # (~11k × 122) or (~11k × 122 × 2)
├── xgb_binary_cw_shap.npy          # ...
├── dnn_binary_cw_shap.npy
├── rf_5class_smote_shap.npy        # (~11k × 122 × 5)
├── xgb_5class_smote_shap.npy
├── dnn_5class_smote_shap.npy
└── feature_importance_rankings.json  # global rankings per model (small, gets committed)
```

Individual SHAP arrays are large (~150 MB each for 5-class) and gitignored. The rankings JSON is small enough to commit.

Downstream notebooks use this data:
- **Notebook 05 (Stability)** — re-computes SHAP under perturbations, compares
- **Notebook 06 (Cross-model agreement)** — applies Krishna et al. 2022's six metrics across models
- **Notebook 07 (SCTS-v2)** — uses stability scores as one input

---
## Session start

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

# Restore git config + credentials from Drive
for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    dst = f'/root/{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        if f == '.git-credentials':
            os.chmod(dst, 0o600)

!git pull
print(f'\n✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
Already up to date.

✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
# Install SHAP if not present
!pip install -q shap==0.46.0
print('✓ SHAP installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 12.4 MB/s eta 0:00:00
✓ SHAP installed


In [3]:
import numpy as np
import pandas as pd
import json, pickle, time
from pathlib import Path

import shap

import torch
import torch.nn as nn

import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'SHAP version: {shap.__version__}')

Device: cpu
SHAP version: 0.46.0


In [4]:
# Paths
PROCESSED = Path(REPO) / 'data' / 'processed' / 'nsl_kdd'
MODELS_DIR = Path(REPO) / 'models' / 'nsl_kdd'
CALIB_DIR = Path(REPO) / 'calibrators' / 'nsl_kdd'
SHAP_DIR = Path(REPO) / 'shap_values' / 'nsl_kdd'
FIG_DIR = Path(REPO) / 'results' / 'figures'
TABLES_DIR = Path(REPO) / 'results' / 'tables'
for d in [SHAP_DIR, FIG_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Load data
X_train = np.load(PROCESSED / 'X_train.npy')
X_test = np.load(PROCESSED / 'X_test.npy')
y_test_b = np.load(PROCESSED / 'y_test_binary.npy')
y_test_5 = np.load(PROCESSED / 'y_test_5class.npy')

# Eval indices from Notebook 03
idx_eval = np.load(CALIB_DIR / 'idx_eval.npy')
X_eval = X_test[idx_eval]
y_eval_b = y_test_b[idx_eval]
y_eval_5 = y_test_5[idx_eval]

# Feature names
with open(PROCESSED / 'feature_names.json') as f:
    FEATURE_NAMES = json.load(f)

with open(PROCESSED / 'class_mappings.json') as f:
    class_info = json.load(f)
INT_TO_CATEGORY = {int(k): v for k, v in class_info['multiclass_5'].items()}
CLASS_NAMES_BINARY = ['Normal', 'Attack']
CLASS_NAMES_5 = [INT_TO_CATEGORY[i] for i in range(5)]

print(f'Eval set:    {X_eval.shape} ({len(X_eval):,} samples)')
print(f'Features:    {len(FEATURE_NAMES)}')
print(f'Train set:   {X_train.shape} (used as SHAP background)')

Eval set:    (11272, 122) (11,272 samples)
Features:    122
Train set:   (125973, 122) (used as SHAP background)


---
## Step 1 — Load all 6 canonical models

In [5]:
class MLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(256, 128, 64, 32), dropout=0.3):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

def load_model(name):
    pkl_path = MODELS_DIR / f'{name}.pkl'
    pt_path  = MODELS_DIR / f'{name}.pt'
    if pkl_path.exists():
        with open(pkl_path, 'rb') as f:
            return ('sklearn', pickle.load(f))
    elif pt_path.exists():
        state = torch.load(pt_path, map_location=DEVICE, weights_only=False)
        model = MLP(state['in_dim'], state['n_classes'],
                    hidden=tuple(state['hidden']), dropout=state['dropout']).to(DEVICE)
        model.load_state_dict(state['state_dict'])
        model.eval()
        return ('torch', model)
    raise FileNotFoundError(name)

CANONICAL = {
    'rf_binary_cw':      {'target': 'binary', 'kind': 'tree',  'lib': 'rf'},
    'xgb_binary_cw':     {'target': 'binary', 'kind': 'tree',  'lib': 'xgb'},
    'dnn_binary_cw':     {'target': 'binary', 'kind': 'dnn',   'lib': 'dnn'},
    'rf_5class_smote':   {'target': '5class', 'kind': 'tree',  'lib': 'rf'},
    'xgb_5class_smote':  {'target': '5class', 'kind': 'tree',  'lib': 'xgb'},
    'dnn_5class_smote':  {'target': '5class', 'kind': 'dnn',   'lib': 'dnn'},
}

print('Loading models...\n')
MODELS = {}
for name in CANONICAL:
    t0 = time.time()
    MODELS[name] = load_model(name)
    print(f'  ✓ {name:<22} ({time.time()-t0:.1f}s)')

Loading models...

  ✓ rf_binary_cw           (1.4s)
  ✓ xgb_binary_cw          (1.2s)
  ✓ dnn_binary_cw          (0.7s)
  ✓ rf_5class_smote        (1.6s)
  ✓ xgb_5class_smote       (0.9s)
  ✓ dnn_5class_smote       (0.5s)


---
## Step 2 — TreeSHAP for tree models

TreeSHAP is fast and exact. We use `feature_perturbation='tree_path_dependent'` (the default for TreeExplainer), which doesn't require background data.

In [6]:
def compute_treeshap(name, model_obj):
    '''Compute SHAP values for a tree model. Returns array of shape:
    - Binary: (n_samples, n_features) — single output
    - 5-class: (n_samples, n_features, n_classes)
    '''
    t0 = time.time()
    print(f'\n--- {name} ---')
    explainer = shap.TreeExplainer(model_obj)
    raw = explainer.shap_values(X_eval)
    elapsed = time.time() - t0

    # Normalise shape. Different SHAP/XGBoost versions return different formats:
    # - sklearn RF binary: list[2] of (n, f) or array (n, f, 2)
    # - sklearn RF multi: list[5] of (n, f) or array (n, f, 5)
    # - XGBoost binary: array (n, f)
    # - XGBoost multi: list[5] of (n, f) or array (n, f, 5)
    if isinstance(raw, list):
        # Stack to (n, f, n_classes)
        shap_arr = np.stack(raw, axis=-1)
    else:
        shap_arr = np.asarray(raw)

    print(f'  Raw type: {type(raw).__name__}, normalised shape: {shap_arr.shape}')
    print(f'  Elapsed: {elapsed:.1f}s')

    # Save
    out_path = SHAP_DIR / f'{name}_shap.npy'
    np.save(out_path, shap_arr.astype(np.float32))
    print(f'  ✓ Saved: {out_path.name}  ({out_path.stat().st_size / 1024**2:.1f} MB)')
    return shap_arr

SHAP_VALUES = {}

# Tree models
for name, info in CANONICAL.items():
    if info['kind'] != 'tree':
        continue
    kind, model_obj = MODELS[name]
    SHAP_VALUES[name] = compute_treeshap(name, model_obj)


--- rf_binary_cw ---
  Raw type: ndarray, normalised shape: (11272, 122, 2)
  Elapsed: 1947.3s
  ✓ Saved: rf_binary_cw_shap.npy  (10.5 MB)

--- xgb_binary_cw ---


ValueError: could not convert string to float: '[5E-1]'

In [7]:
!pip install -q shap==0.47.2 --upgrade
print('✓ SHAP upgraded')

# Re-import to pick up new version
import importlib, shap
importlib.reload(shap)
print(f'SHAP version: {shap.__version__}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.2 MB/s eta 0:00:00
✓ SHAP upgraded
SHAP version: 0.46.0


In [8]:
import numpy as np
import time

# Stratified subsample of eval set for RF (XGBoost is fast enough on full eval)
RF_SAMPLE_SIZE = 3000
rng = np.random.RandomState(42)

# y_eval_5 already loaded from notebook
rf_idx = []
per_class = RF_SAMPLE_SIZE // 5
for c in range(5):
    class_pool = np.where(y_eval_5 == c)[0]
    n_take = min(per_class, len(class_pool))
    rf_idx.extend(rng.choice(class_pool, n_take, replace=False).tolist())
rf_idx = np.array(rf_idx)
X_eval_rf = X_eval[rf_idx]
print(f'RF subsample: {X_eval_rf.shape}')

# Save the rf subsample indices so downstream notebooks know which eval rows the RF SHAP corresponds to
np.save(SHAP_DIR / 'rf_subsample_idx.npy', rf_idx)

# Re-run TreeSHAP on tree models — RF on subsample, XGB on full eval
for name, info in CANONICAL.items():
    if info['kind'] != 'tree':
        continue
    if name.startswith('rf'):
        X_for_shap = X_eval_rf
    else:
        X_for_shap = X_eval

    print(f'\n--- {name} ---  (n={len(X_for_shap):,})')
    t0 = time.time()
    kind, model_obj = MODELS[name]
    explainer = shap.TreeExplainer(model_obj)
    raw = explainer.shap_values(X_for_shap)
    elapsed = time.time() - t0

    if isinstance(raw, list):
        shap_arr = np.stack(raw, axis=-1)
    else:
        shap_arr = np.asarray(raw)

    print(f'  Shape: {shap_arr.shape}  Elapsed: {elapsed:.1f}s')

    out_path = SHAP_DIR / f'{name}_shap.npy'
    np.save(out_path, shap_arr.astype(np.float32))
    SHAP_VALUES[name] = shap_arr
    print(f'  ✓ Saved: {out_path.name}  ({out_path.stat().st_size / 1024**2:.1f} MB)')

RF subsample: (2433, 122)

--- rf_binary_cw ---  (n=2,433)
  Shape: (2433, 122, 2)  Elapsed: 419.2s
  ✓ Saved: rf_binary_cw_shap.npy  (2.3 MB)

--- xgb_binary_cw ---  (n=11,272)


ValueError: could not convert string to float: '[5E-1]'

In [11]:
import xgboost as xgb
import numpy as np
import time

def compute_xgb_native_shap(name, model_obj, X):
    """Use XGBoost's built-in SHAP (pred_contribs) instead of shap.TreeExplainer."""
    t0 = time.time()
    print(f'\n--- {name} ---  (n={len(X):,})')

    # XGBoost's sklearn wrapper exposes the underlying Booster
    booster = model_obj.get_booster()

    # pred_contribs=True returns SHAP values with shape:
    # - Binary: (n_samples, n_features + 1)  [last column is the bias term]
    # - Multi-class: (n_samples, n_classes, n_features + 1)
    dmatrix = xgb.DMatrix(X)
    shap_raw = booster.predict(dmatrix, pred_contribs=True)
    elapsed = time.time() - t0

    # Drop the bias column (last index along the features axis)
    # and reshape to match TreeShap output format (n, f) or (n, f, c)
    if shap_raw.ndim == 2:
        # Binary: shape (n, f+1)
        shap_arr = shap_raw[:, :-1]  # drop bias
        # Wrap in 2-class structure to match RF binary: (n, f, 2) where class 1 is the contribution
        shap_arr = np.stack([-shap_arr, shap_arr], axis=-1)
    else:
        # Multi-class: shape (n, n_classes, f+1)
        shap_arr = shap_raw[:, :, :-1]
        # Transpose to (n, f, n_classes) to match our convention
        shap_arr = np.transpose(shap_arr, (0, 2, 1))

    print(f'  Raw shape: {shap_raw.shape} → normalised: {shap_arr.shape}')
    print(f'  Elapsed: {elapsed:.1f}s')

    out_path = SHAP_DIR / f'{name}_shap.npy'
    np.save(out_path, shap_arr.astype(np.float32))
    print(f'  ✓ Saved: {out_path.name}  ({out_path.stat().st_size / 1024**2:.1f} MB)')
    return shap_arr

# Run for both XGBoost models
for name in ['xgb_binary_cw', 'xgb_5class_smote']:
    info = CANONICAL[name]
    _, model_obj = MODELS[name]
    SHAP_VALUES[name] = compute_xgb_native_shap(name, model_obj, X_eval)

print('\n✓ XGBoost SHAP done via native pred_contribs')


--- xgb_binary_cw ---  (n=11,272)
  Raw shape: (11272, 123) → normalised: (11272, 122, 2)
  Elapsed: 47.8s
  ✓ Saved: xgb_binary_cw_shap.npy  (10.5 MB)

--- xgb_5class_smote ---  (n=11,272)
  Raw shape: (11272, 5, 123) → normalised: (11272, 122, 5)
  Elapsed: 195.5s
  ✓ Saved: xgb_5class_smote_shap.npy  (26.2 MB)

✓ XGBoost SHAP done via native pred_contribs


---
## Step 3 — DeepSHAP for DNN models

DeepSHAP uses a background distribution to approximate Shapley values for deep networks. We sample 500 background points from the training set (standard practice — more is better but diminishing returns).

In [12]:
BACKGROUND_SIZE = 500

# Stratified sample of background data so all classes represented
y_train_5 = np.load(PROCESSED / 'y_train_5class.npy')
rng = np.random.RandomState(SEED)
bg_indices = []
per_class = BACKGROUND_SIZE // 5
for c in range(5):
    class_idx = np.where(y_train_5 == c)[0]
    n_sample = min(per_class, len(class_idx))
    bg_indices.extend(rng.choice(class_idx, n_sample, replace=False).tolist())
bg_indices = np.array(bg_indices)
X_background = X_train[bg_indices]
print(f'Background set: {X_background.shape} ({BACKGROUND_SIZE} stratified samples)')

def compute_deepshap(name, model_obj):
    '''Compute SHAP values for a DNN. Returns (n_samples, n_features, n_classes).'''
    t0 = time.time()
    print(f'\n--- {name} ---')

    bg_tensor = torch.tensor(X_background, dtype=torch.float32).to(DEVICE)
    eval_tensor = torch.tensor(X_eval, dtype=torch.float32).to(DEVICE)

    # DeepExplainer works on PyTorch models in eval mode
    model_obj.eval()
    explainer = shap.DeepExplainer(model_obj, bg_tensor)

    # Returns list of length n_classes, each (n_samples, n_features)
    # OR a single (n_samples, n_features) for binary
    raw = explainer.shap_values(eval_tensor, check_additivity=False)
    elapsed = time.time() - t0

    if isinstance(raw, list):
        shap_arr = np.stack(raw, axis=-1)
    else:
        shap_arr = np.asarray(raw)

    print(f'  Raw type: {type(raw).__name__}, normalised shape: {shap_arr.shape}')
    print(f'  Elapsed: {elapsed:.1f}s')

    out_path = SHAP_DIR / f'{name}_shap.npy'
    np.save(out_path, shap_arr.astype(np.float32))
    print(f'  ✓ Saved: {out_path.name}  ({out_path.stat().st_size / 1024**2:.1f} MB)')
    return shap_arr

for name, info in CANONICAL.items():
    if info['kind'] != 'dnn':
        continue
    kind, model_obj = MODELS[name]
    SHAP_VALUES[name] = compute_deepshap(name, model_obj)

Background set: (452, 122) (500 stratified samples)

--- dnn_binary_cw ---
  Raw type: ndarray, normalised shape: (11272, 122, 2)
  Elapsed: 837.7s
  ✓ Saved: dnn_binary_cw_shap.npy  (10.5 MB)

--- dnn_5class_smote ---
  Raw type: ndarray, normalised shape: (11272, 122, 5)
  Elapsed: 2061.5s
  ✓ Saved: dnn_5class_smote_shap.npy  (26.2 MB)


In [13]:
import time
print('\n--- rf_5class_smote ---')
t0 = time.time()
kind, model_obj = MODELS['rf_5class_smote']
explainer = shap.TreeExplainer(model_obj)
raw = explainer.shap_values(X_eval_rf)
elapsed = time.time() - t0

if isinstance(raw, list):
    shap_arr = np.stack(raw, axis=-1)
else:
    shap_arr = np.asarray(raw)

print(f'  Shape: {shap_arr.shape}  Elapsed: {elapsed:.1f}s')
out_path = SHAP_DIR / 'rf_5class_smote_shap.npy'
np.save(out_path, shap_arr.astype(np.float32))
SHAP_VALUES['rf_5class_smote'] = shap_arr
print(f'  ✓ Saved')


--- rf_5class_smote ---
  Shape: (2433, 122, 5)  Elapsed: 716.2s
  ✓ Saved


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, json, pickle, time
import numpy as np
import pandas as pd
from pathlib import Path

# Restore git creds
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)
for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

# Install SHAP again (Colab reset wipes packages)
!pip install -q shap==0.47.2

import shap
import torch
import torch.nn as nn

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'SHAP: {shap.__version__}')

# Paths
PROCESSED = Path(REPO) / 'data' / 'processed' / 'nsl_kdd'
MODELS_DIR = Path(REPO) / 'models' / 'nsl_kdd'
CALIB_DIR = Path(REPO) / 'calibrators' / 'nsl_kdd'
SHAP_DIR = Path(REPO) / 'shap_values' / 'nsl_kdd'
FIG_DIR = Path(REPO) / 'results' / 'figures'
TABLES_DIR = Path(REPO) / 'results' / 'tables'

# Load data
X_train = np.load(PROCESSED / 'X_train.npy')
X_test = np.load(PROCESSED / 'X_test.npy')
y_test_b = np.load(PROCESSED / 'y_test_binary.npy')
y_test_5 = np.load(PROCESSED / 'y_test_5class.npy')
idx_eval = np.load(CALIB_DIR / 'idx_eval.npy')
X_eval = X_test[idx_eval]
y_eval_b = y_test_b[idx_eval]
y_eval_5 = y_test_5[idx_eval]

with open(PROCESSED / 'feature_names.json') as f:
    FEATURE_NAMES = json.load(f)
with open(PROCESSED / 'class_mappings.json') as f:
    class_info = json.load(f)
INT_TO_CATEGORY = {int(k): v for k, v in class_info['multiclass_5'].items()}
CLASS_NAMES_BINARY = ['Normal', 'Attack']
CLASS_NAMES_5 = [INT_TO_CATEGORY[i] for i in range(5)]

# MLP class definition (needed to load DNN models)
class MLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(256, 128, 64, 32), dropout=0.3):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

def load_model(name):
    pkl_path = MODELS_DIR / f'{name}.pkl'
    pt_path = MODELS_DIR / f'{name}.pt'
    if pkl_path.exists():
        with open(pkl_path, 'rb') as f:
            return ('sklearn', pickle.load(f))
    elif pt_path.exists():
        state = torch.load(pt_path, map_location=DEVICE, weights_only=False)
        model = MLP(state['in_dim'], state['n_classes'],
                    hidden=tuple(state['hidden']), dropout=state['dropout']).to(DEVICE)
        model.load_state_dict(state['state_dict'])
        model.eval()
        return ('torch', model)
    raise FileNotFoundError(name)

# Load just the two DNN models (we don't need to reload tree models — their SHAP is saved)
MODELS = {}
for name in ['dnn_binary_cw', 'dnn_5class_smote']:
    MODELS[name] = load_model(name)

print(f'\n✓ Environment restored')
print(f'X_eval: {X_eval.shape}')
print(f'DNN models loaded: {list(MODELS.keys())}')

# Verify SHAP files for tree models still exist on Drive (they should — saved earlier)
print(f'\nExisting SHAP files:')
for f in sorted(SHAP_DIR.iterdir()):
    if f.is_file():
        size_mb = f.stat().st_size / 1024**2
        print(f'  {f.name}  ({size_mb:.1f} MB)')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.3 MB/s eta 0:00:00
Device: cpu
SHAP: 0.47.2

✓ Environment restored
X_eval: (11272, 122)
DNN models loaded: ['dnn_binary_cw', 'dnn_5class_smote']

Existing SHAP files:
  dnn_5class_smote_shap.npy  (26.2 MB)
  dnn_binary_cw_shap.npy  (10.5 MB)
  rf_5class_smote_shap.npy  (5.7 MB)
  rf_binary_cw_shap.npy  (2.3 MB)
  rf_subsample_idx.npy  (0.0 MB)
  xgb_5class_smote_shap.npy  (26.2 MB)
  xgb_binary_cw_shap.npy  (10.5 MB)


In [9]:
# Load all 6 SHAP arrays from disk
SHAP_VALUES = {}
CANONICAL = {
    'rf_binary_cw':      {'target': 'binary', 'kind': 'tree'},
    'xgb_binary_cw':     {'target': 'binary', 'kind': 'tree'},
    'dnn_binary_cw':     {'target': 'binary', 'kind': 'dnn'},
    'rf_5class_smote':   {'target': '5class', 'kind': 'tree'},
    'xgb_5class_smote':  {'target': '5class', 'kind': 'tree'},
    'dnn_5class_smote':  {'target': '5class', 'kind': 'dnn'},
}

print('Loading saved SHAP arrays:\n')
for name in CANONICAL:
    arr = np.load(SHAP_DIR / f'{name}_shap.npy')
    SHAP_VALUES[name] = arr
    print(f'  {name:<22} {arr.shape}  ({arr.nbytes/1024**2:.1f} MB)')

# ============== STEP 5: Global rankings ==============
RANKINGS = {}
for name, info in CANONICAL.items():
    sv = SHAP_VALUES[name]
    target = info['target']
    rankings = {}

    if target == 'binary':
        if sv.ndim == 2:
            mean_abs = np.abs(sv).mean(axis=0)
        else:
            mean_abs = np.abs(sv[:, :, 1]).mean(axis=0)
        order = np.argsort(-mean_abs)
        rankings['overall'] = [
            {'feature': FEATURE_NAMES[i], 'mean_abs_shap': float(mean_abs[i])}
            for i in order[:20]
        ]
    else:
        for c in range(5):
            mean_abs = np.abs(sv[:, :, c]).mean(axis=0)
            order = np.argsort(-mean_abs)
            rankings[CLASS_NAMES_5[c]] = [
                {'feature': FEATURE_NAMES[i], 'mean_abs_shap': float(mean_abs[i])}
                for i in order[:20]
            ]
        mean_abs_overall = np.abs(sv).mean(axis=(0, 2))
        order = np.argsort(-mean_abs_overall)
        rankings['overall'] = [
            {'feature': FEATURE_NAMES[i], 'mean_abs_shap': float(mean_abs_overall[i])}
            for i in order[:20]
        ]
    RANKINGS[name] = rankings

with open(SHAP_DIR / 'feature_importance_rankings.json', 'w') as f:
    json.dump(RANKINGS, f, indent=2)

print('\n=== Top 10 overall features per model ===\n')
for name in CANONICAL:
    print(f'{name}:')
    for i, item in enumerate(RANKINGS[name]['overall'][:10]):
        print(f'  {i+1:2}. {item["feature"]:<35} {item["mean_abs_shap"]:.4f}')
    print()

# ============== STEP 6: Cross-architecture overlap ==============
def top_k(ranking_list, k=10):
    return set(item['feature'] for item in ranking_list[:k])

print('=== Top-10 cross-architecture overlap ===\n')
print('Binary models:')
rf_b = top_k(RANKINGS['rf_binary_cw']['overall'])
xgb_b = top_k(RANKINGS['xgb_binary_cw']['overall'])
dnn_b = top_k(RANKINGS['dnn_binary_cw']['overall'])
print(f'  RF ∩ XGB:  {len(rf_b & xgb_b)} features → {sorted(rf_b & xgb_b)}')
print(f'  RF ∩ DNN:  {len(rf_b & dnn_b)} features → {sorted(rf_b & dnn_b)}')
print(f'  XGB ∩ DNN: {len(xgb_b & dnn_b)} features → {sorted(xgb_b & dnn_b)}')
print(f'  All three: {len(rf_b & xgb_b & dnn_b)} features → {sorted(rf_b & xgb_b & dnn_b)}')

print('\n5-class models (overall):')
rf_5 = top_k(RANKINGS['rf_5class_smote']['overall'])
xgb_5 = top_k(RANKINGS['xgb_5class_smote']['overall'])
dnn_5 = top_k(RANKINGS['dnn_5class_smote']['overall'])
print(f'  RF ∩ XGB:  {len(rf_5 & xgb_5)} features')
print(f'  RF ∩ DNN:  {len(rf_5 & dnn_5)} features')
print(f'  XGB ∩ DNN: {len(xgb_5 & dnn_5)} features')
print(f'  All three: {len(rf_5 & xgb_5 & dnn_5)} features → {sorted(rf_5 & xgb_5 & dnn_5)}')

Loading saved SHAP arrays:

  rf_binary_cw           (2433, 122, 2)  (2.3 MB)
  xgb_binary_cw          (11272, 122, 2)  (10.5 MB)
  dnn_binary_cw          (11272, 122, 2)  (10.5 MB)
  rf_5class_smote        (2433, 122, 5)  (5.7 MB)
  xgb_5class_smote       (11272, 122, 5)  (26.2 MB)
  dnn_5class_smote       (11272, 122, 5)  (26.2 MB)

=== Top 10 overall features per model ===

rf_binary_cw:
   1. src_bytes                           0.0850
   2. dst_bytes                           0.0504
   3. flag_SF                             0.0387
   4. dst_host_srv_count                  0.0362
   5. logged_in                           0.0312
   6. count                               0.0288
   7. dst_host_same_srv_rate              0.0280
   8. same_srv_rate                       0.0267
   9. dst_host_diff_srv_rate              0.0219
  10. dst_host_rerror_rate                0.0217

xgb_binary_cw:
   1. src_bytes                           3.8849
   2. dst_host_srv_count                  1.1858
  

---
## Step 4 — Sanity check on shapes

Each SHAP array should have a predictable shape based on target type.

In [8]:
print(f'Expected: binary → ({len(X_eval):,}, {len(FEATURE_NAMES)}) or (..., 2)')
print(f'Expected: 5class → ({len(X_eval):,}, {len(FEATURE_NAMES)}, 5)')
print()
print(f'{"Model":<22} {"Shape":<25} {"Dtype":<12} {"Size"}')
print('-' * 65)
for name in CANONICAL:
    sv = SHAP_VALUES[name]
    size_mb = sv.nbytes / 1024**2
    print(f'{name:<22} {str(sv.shape):<25} {str(sv.dtype):<12} {size_mb:>6.1f} MB')

Expected: binary → (11,272, 122) or (..., 2)
Expected: 5class → (11,272, 122, 5)

Model                  Shape                     Dtype        Size
-----------------------------------------------------------------


NameError: name 'CANONICAL' is not defined

---
## Step 5 — Global feature importance rankings

For each model, **global importance** of a feature = mean of absolute SHAP values across samples.

For 5-class, we get per-class rankings (one ranking per attack type) plus an overall ranking (averaged across classes).

In [ ]:
RANKINGS = {}

for name, info in CANONICAL.items():
    sv = SHAP_VALUES[name]
    target = info['target']
    rankings = {}

    if target == 'binary':
        # Binary: shape is (n, f) or (n, f, 2)
        if sv.ndim == 2:
            # Single output — common for XGB binary
            mean_abs = np.abs(sv).mean(axis=0)
        else:
            # 2-output — common for sklearn RF binary; use class 1 (Attack)
            mean_abs = np.abs(sv[:, :, 1]).mean(axis=0)
        # Top 20
        order = np.argsort(-mean_abs)
        rankings['overall'] = [
            {'feature': FEATURE_NAMES[i], 'mean_abs_shap': float(mean_abs[i])}
            for i in order[:20]
        ]
    else:
        # 5-class: shape is (n, f, 5)
        assert sv.ndim == 3 and sv.shape[2] == 5
        for c in range(5):
            mean_abs = np.abs(sv[:, :, c]).mean(axis=0)
            order = np.argsort(-mean_abs)
            rankings[CLASS_NAMES_5[c]] = [
                {'feature': FEATURE_NAMES[i], 'mean_abs_shap': float(mean_abs[i])}
                for i in order[:20]
            ]
        # Overall — average across classes
        mean_abs_overall = np.abs(sv).mean(axis=(0, 2))
        order = np.argsort(-mean_abs_overall)
        rankings['overall'] = [
            {'feature': FEATURE_NAMES[i], 'mean_abs_shap': float(mean_abs_overall[i])}
            for i in order[:20]
        ]

    RANKINGS[name] = rankings

# Save
with open(SHAP_DIR / 'feature_importance_rankings.json', 'w') as f:
    json.dump(RANKINGS, f, indent=2)

# Show top 10 overall for each model
print('Top 10 overall features per model:\n')
for name in CANONICAL:
    print(f'\n{name}:')
    for i, item in enumerate(RANKINGS[name]['overall'][:10]):
        print(f'  {i+1:2}. {item["feature"]:<35} {item["mean_abs_shap"]:.4f}')

---
## Step 6 — Top-K agreement across architectures (preview for Notebook 06)

How much do RF, XGBoost, and DNN agree on which features matter? This is the basic question Notebook 06 will answer rigorously using Krishna et al. 2022's six metrics. Here we just preview the overlap of top-10 features.

In [ ]:
def top_k(ranking_list, k=10):
    return set(item['feature'] for item in ranking_list[:k])

print('Top-10 feature overlap (binary models):')
rf_b = top_k(RANKINGS['rf_binary_cw']['overall'])
xgb_b = top_k(RANKINGS['xgb_binary_cw']['overall'])
dnn_b = top_k(RANKINGS['dnn_binary_cw']['overall'])
print(f'  RF ∩ XGB:  {len(rf_b & xgb_b)} features → {sorted(rf_b & xgb_b)}')
print(f'  RF ∩ DNN:  {len(rf_b & dnn_b)} features → {sorted(rf_b & dnn_b)}')
print(f'  XGB ∩ DNN: {len(xgb_b & dnn_b)} features → {sorted(xgb_b & dnn_b)}')
print(f'  All three: {len(rf_b & xgb_b & dnn_b)} features → {sorted(rf_b & xgb_b & dnn_b)}')

print('\nTop-10 feature overlap (5-class models, overall):')
rf_5 = top_k(RANKINGS['rf_5class_smote']['overall'])
xgb_5 = top_k(RANKINGS['xgb_5class_smote']['overall'])
dnn_5 = top_k(RANKINGS['dnn_5class_smote']['overall'])
print(f'  RF ∩ XGB:  {len(rf_5 & xgb_5)} features')
print(f'  RF ∩ DNN:  {len(rf_5 & dnn_5)} features')
print(f'  XGB ∩ DNN: {len(xgb_5 & dnn_5)} features')
print(f'  All three: {len(rf_5 & xgb_5 & dnn_5)} features → {sorted(rf_5 & xgb_5 & dnn_5)}')

---
## Step 7 — Visualisations for the paper

Two key figures:
1. Top-10 feature importance bar chart per model (paper-ready)
2. Top-10 overlap heatmap across model pairs

In [ ]:
# Figure 1: top-10 importance for each model (binary on top, 5-class on bottom)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
model_order = ['rf_binary_cw', 'xgb_binary_cw', 'dnn_binary_cw',
                'rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote']
for ax, name in zip(axes.flatten(), model_order):
    top10 = RANKINGS[name]['overall'][:10]
    features = [item['feature'] for item in top10]
    values = [item['mean_abs_shap'] for item in top10]
    bars = ax.barh(range(len(features)), values, color='#4C72B0')
    ax.set_yticks(range(len(features))); ax.set_yticklabels(features, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Mean |SHAP|')
    ax.set_title(name, fontsize=11)
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'nslkdd_top10_features_per_model.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2: pairwise top-10 overlap heatmap
models = list(CANONICAL.keys())
n_models = len(models)
overlap_matrix = np.zeros((n_models, n_models), dtype=int)
for i, mi in enumerate(models):
    ti = top_k(RANKINGS[mi]['overall'])
    for j, mj in enumerate(models):
        tj = top_k(RANKINGS[mj]['overall'])
        overlap_matrix[i, j] = len(ti & tj)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(overlap_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=models, yticklabels=models, cbar_kws={'label': 'Top-10 overlap'},
            ax=ax, vmin=0, vmax=10)
ax.set_title('Top-10 SHAP feature overlap between models')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / 'nslkdd_top10_overlap_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Save summary table for paper

In [ ]:
# Build summary CSV: top-10 features per model with their mean |SHAP|
rows = []
for name in CANONICAL:
    for rank, item in enumerate(RANKINGS[name]['overall'][:10], 1):
        rows.append({
            'Model': name,
            'Rank': rank,
            'Feature': item['feature'],
            'Mean_abs_SHAP': item['mean_abs_shap'],
        })

df = pd.DataFrame(rows)
df.to_csv(TABLES_DIR / 'nslkdd_top10_shap_features.csv', index=False)
print(f'✓ Saved: {TABLES_DIR / "nslkdd_top10_shap_features.csv"}')
print(f'\nFirst 10 rows:')
print(df.head(15).to_string(index=False))

---
## Step 9 — Commit

The big SHAP .npy arrays are gitignored. Only the rankings JSON, summary CSV, and figures get committed.

In [ ]:
os.chdir(REPO)
!git add notebooks/04_shap_computation.ipynb
!git add shap_values/nsl_kdd/feature_importance_rankings.json
!git add results/
!git status --short
!git commit -m 'Notebook 04: SHAP values for all 6 calibrated models'
!git push

---
## Summary

**What this notebook produced:**
- ✓ 6 SHAP value arrays (one per model, in `shap_values/nsl_kdd/`)
- ✓ Global feature importance rankings (`feature_importance_rankings.json`)
- ✓ Top-10 features summary CSV (paper table)
- ✓ Two figures (top-10 importance bars, top-10 overlap heatmap)
- ✓ Cross-architecture overlap preview

**What to look at:**
- The top-10 feature lists — do RF, XGBoost, DNN agree on what matters? Some agreement is expected; full agreement would suggest the models are learning the same thing (boring for the paper); strong disagreement is a finding.
- The overlap heatmap — diagonal is always 10 (self-overlap), off-diagonal numbers tell the story. 5-8 is typical, 8-10 is high agreement, 0-4 is divergence.

**Next notebook (05):** Stability testing — re-compute SHAP under three perturbation types (Gaussian noise, FGSM, PGD) and measure how much the rankings shift. This is novelty C3 of the paper.
